This notebook invokes the Shared, MWC, and Public APIs to deploy Healthcare Data Solutions Capabilities. 
To use this notebook:
    1. Create a workspace in Fabric
    2. Import the notebook
    3. Run All

In [ ]:
# imports random module
import random
 
# Generates a random number between
# a given positive range
r1 = random.randint(1, 100)

# Create Healthcare Data Solutions Artifact
solution_name = f"healthcare{r1}"

In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from trident_token_library_wrapper import PyTridentTokenLibrary
from http import HTTPStatus
import requests
import gson
import uuid
import json
import time

_PBI_GLOBAL_SERVICE_ENDPOINTS = {
"public": "https://api.powerbi.com/",
"fairfax": "https://api.powerbigov.us",
"mooncake": "https://api.powerbi.cn",
"blackforest": "https://app.powerbi.de",
"msit": "https://api.powerbi.com/",
"prod": "https://api.powerbi.com/",
"int3": "https://biazure-int-edog-redirect.analysis-df.windows.net/",
"dxt": "https://powerbistagingapi.analysis.windows.net/",
"edog": "https://biazure-int-edog-redirect.analysis-df.windows.net/",
"dev": "https://onebox-redirect.analysis.windows-int.net/",
"console": "http://localhost:5001/",
"daily": "https://dailyapi.powerbi.com/",
}


_DEFAULT_GLOBAL_SERVICE_ENDPOINT = "https://api.powerbi.com/"
_FETCH_CLUSTER_DETAIL_URI = "powerbi/globalservice/v201606/clusterDetails"

class MwcToken:
    def __init__(
        self, TargetUriHost: str = "", CapacityObjectId: str = "", Token: str = ""
    ):
        self.TargetUriHost = TargetUriHost
        self.CapacityObjectId = CapacityObjectId
        self.Token = Token    

class SharedService:
    """
    Used to make calls to the Shared API
    """
    def __init__(self, spark):
        self.spark = spark
        self.pbienv = spark.sparkContext.getConf().get("spark.trident.pbienv", "").lower()

    def get_aad_token(self):
        return PyTridentTokenLibrary.get_access_token("pbi")

    def generate_mwc_token(self, capacity_id: str, workspace_id: str):
        """
        Generate MWC Token
        """
        shared_url = self._get_shared_host()
        mwc_token_url = f"{shared_url}/metadata/v201606/generatemwctokenv2"
        return self._get_hds_mwc_token(mwc_token_url, capacity_id, workspace_id)

    def create_hds_artifact(self, workspace_id: str, display_name: str):
        """
        Make a post call to create HDS artifact
        """
        shared_url = self._get_shared_host()
        deploy_hds_capability_url = f"{shared_url}/metadata/workspaces/{workspace_id}/artifacts"

        pay_load = {
            "artifactType": "HealthDataManager",
            "displayName": f"{display_name}",
            "description": "Test DMH Artifact creation."
        }

        resp = self._make_http_post_call(deploy_hds_capability_url, pay_load)
        if resp.status_code != HTTPStatus.OK:
                raise Exception(
                    f"Failed to deploy Healthcare Data Solutions Artifact {resp.status_code}:{resp.content}: {resp.headers}"
                )
        
        print(f"Successfully deployed Healthcare Data Solutions Artifact - {display_name}\n")

        resp_content = json.loads(resp.content)
        return resp_content["objectId"]

    def _make_http_post_call(self, url, pay_load = None):

        if pay_load is None:
            pay_load = {}

        headers = {}
        headers["Authorization"] = f"Bearer {self.get_aad_token()}"
        headers["RequestId"] = str(uuid.uuid4())

        resp = requests.post(url=url, json=pay_load, headers=headers)
        return resp

    def _make_http_get_call(self, url):
        headers = {}
        headers["Authorization"] = f"Bearer {self.get_aad_token()}"
        headers["RequestId"] = str(uuid.uuid4())

        resp = requests.get(url=url, headers=headers)
        return resp
                
    def _get_hds_mwc_token(self, url: str, capacity_id: str, workspace_id: str, workload_type: str = "dmh"):
        pay_load = {
            "capacityObjectId": capacity_id,
            "workspaceObjectId": workspace_id,
            "workloadType": workload_type,
        }

        try:
            resp = self._make_http_post_call(url=url, pay_load=pay_load)
            if resp.status_code != HTTPStatus.OK:
                raise Exception(
                    f"Failed to generate mwc token {resp.status_code}:{resp.content}"
                )
            
            res, e = gson.unmarshal_from_str(resp.content, MwcToken)
            if e:
                raise e
            res.TargetUriHost = "https://" + res.TargetUriHost
            return res
        except Exception:
            print("Failed to generate MWC Token")
            raise

    def _get_shared_host(self):
        url = _PBI_GLOBAL_SERVICE_ENDPOINTS.get(self.pbienv, _DEFAULT_GLOBAL_SERVICE_ENDPOINT) + _FETCH_CLUSTER_DETAIL_URI

        try:
            resp = self._make_http_get_call(url=url)
            if resp.status_code != HTTPStatus.OK:
                raise Exception(
                    f"Fetch cluster details returns {resp.status_code}:{resp.content}"
                )
            resp_body = json.loads(resp.content)
            return resp_body["clusterUrl"]
        except Exception:
                print("Failed to fetch shared hos")
                raise


In [ ]:
class HdsWorkloadService:

    """
    HDS Workload Service to invoke HDS Workload Endpoints
    """
    def __init__(self, spark, capacity_id, mwc_host, solution_name):
        self.spark = spark
        self.capacity_id = capacity_id
        self.mwc_host = mwc_host
        self.solution_name = solution_name
    
    def ping_workload_capability(self, mwc_token):
        """
        Ping HDS Workload Capability
        """
        url = f"{self.mwc_host}/webapi/capacities/{self.capacity_id}/workloads/dmh/DMHService/automatic/capabilities/ping"
        headers = {}
        headers["Authorization"] = f"MwcToken {mwc_token}"
        resp = requests.get(url=url, headers=headers)
        from http import HTTPStatus

        if resp.status_code != HTTPStatus.OK:
            raise Exception(
                f"Failed to ping workload capability {resp.status_code}:{resp.content}"
            )
            
        return resp.content

    def list_workload_capability(self, mwc_token, artifact_id):
        """
        List all capabilties in workload
        """
        url = f"{self.mwc_host}/webapi/capacities/{self.capacity_id}/workloads/dmh/DMHService/automatic/capabilities"

        headers = {}
        headers["Authorization"] = f"MwcToken {mwc_token}"
        headers["x-ms-workload-resource-moniker"] = artifact_id

        resp = requests.get(url=url, headers=headers)
        from http import HTTPStatus

        if resp.status_code != HTTPStatus.OK:
            raise Exception(
                f"Failed to ping workload capability {resp.status_code}:{resp.content}"
            )
            
        capabilities = json.loads(resp.content)
        return capabilities

    def deploy_all_capabilities(self, mwc_token, artifact_id):
        """
        Deploy all capabilties in workload
        """
        capabilities = self.list_workload_capability(mwc_token, artifact_id)
        base_capabilities = ["relational-fhir-data-foundations"]
        for base_capability in base_capabilities:
            self._call_deploy_capability(mwc_token, artifact_id, base_capability, self.solution_name)
        for capability in capabilities:
            if capability["name"] not in base_capabilities:
                self._call_deploy_capability(mwc_token, artifact_id, capability["name"], self.solution_name)

        print("Finished attempt to deploy all capabilities. Please check above for any failures.")

    def _call_deploy_capability(self, mwc_token, artifact_id, capability_key, prefix):
        """
        Deploy a given capability
        """
        print(f"Deploying Capability - {capability_key}")
        try:
            self.deploy_workload_capability(mwc_token, artifact_id, capability_key, self.solution_name)
            time.sleep(10)
        except Exception as err:
            print(f"ERROR: Failed to deploy {capability_key}: ", err)
        print("\n")

    def deploy_workload_capability(self, mwc_token, artifact_id, capability_key, prefix):
        """
        Ping HDS Workload Capability
        """
        url = f"{self.mwc_host}/webapi/capacities/{self.capacity_id}/workloads/dmh/DMHService/automatic/artifacts/{artifact_id}/capabilities"

        pay_load = {
            "capabilityKey": f"{capability_key}",
            "uniquePrefix": f"{prefix}"
        }

        headers = {}
        headers["Authorization"] = f"MwcToken {mwc_token}"
        headers["x-ms-workload-resource-moniker"] = artifact_id
        resp = requests.post(url=url, json=pay_load, headers=headers)
        from http import HTTPStatus

        if resp.status_code != HTTPStatus.OK:
            raise Exception(
                f"Failed to ping workload capability {resp.status_code}:{resp.content}:{resp.headers}"
            )
            
        print(f"Successfully deployed capability {capability_key}. {resp.status_code}:{resp.content}:{resp.headers}")
        return resp.content

In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from trident_token_library_wrapper import PyTridentTokenLibrary
from http import HTTPStatus
import requests
import gson
import uuid
import json
import time


_PUBLIC_API_ENDPOINT = "https://api.fabric.microsoft.com/"

class PublicAPIService:
    """
    Used to make calls to the Public API
    """
    def __init__(self, spark, workspace_id):
        self.spark = spark
        self.pbienv = spark.sparkContext.getConf().get("spark.trident.pbienv", "").lower()
        self.workspace_id = workspace_id

    def get_aad_token(self):
        return PyTridentTokenLibrary.get_access_token("pbi")

    def _make_http_post_call(self, url, pay_load = None):

        if pay_load is None:
            pay_load = {}

        headers = {}
        headers["Authorization"] = f"Bearer {self.get_aad_token()}"
        headers["RequestId"] = str(uuid.uuid4())

        resp = requests.post(url=url, json=pay_load, headers=headers)
        return resp

    def _make_http_get_call(self, url):

        headers = {}
        headers["Authorization"] = f"Bearer {self.get_aad_token()}"
        headers["RequestId"] = str(uuid.uuid4())

        resp = requests.get(url=url, headers=headers)
        return resp

    def get_notebooks_in_workspace(self):
        """
        Get all notebooks in workspace
        """
        return self.list_items_in_workspace("Notebook")

    def list_items_in_workspace(self, type: str):
        url = f"{_PUBLIC_API_ENDPOINT}/v1/workspaces/{self.workspace_id}/items?type={type}"

        try:
            resp = self._make_http_get_call(url=url)
            if resp.status_code != HTTPStatus.OK:
                raise Exception(
                    f"Failed to list items of {type} in {self.workspace_id}. {resp.status_code}:{resp.content}"
                )
            resp_body = json.loads(resp.content)
            return resp_body
        except Exception:
                print(f"Failed to list items of {type} in {self.workspace_id}")
                raise

In [ ]:
capacity_id = spark.sparkContext._jsc.hadoopConfiguration().get("trident.capacity.id", "")
workspace_id = spark.sparkContext._jsc.hadoopConfiguration().get("trident.artifact.workspace.id", "")

# Create Healthcare Data Solutions Artifact
shared_service = SharedService(spark)
artifact_id = shared_service.create_hds_artifact(workspace_id, solution_name)

# Invoke Workloads
mwc_token = shared_service.generate_mwc_token(capacity_id, workspace_id)
hds_workload_service = HdsWorkloadService(spark, capacity_id, mwc_token.TargetUriHost, solution_name)
print(hds_workload_service.ping_workload_capability(mwc_token.Token))
hds_workload_service.deploy_all_capabilities(mwc_token.Token, artifact_id)

# Use public APIs to retrieve the notebooks deployed
public_api_service = PublicAPIService(spark,workspace_id)
print(public_api_service.get_notebooks_in_workspace())